In [ ]:
from google.colab import userdata
import os
os.environ['OPENAI_API_KEY']=userdata.get('OPENAI_API_KEY')

In [ ]:
def get_completion(prompt, model="gpt-4o"):
    """
    Sends a single prompt to the specified OpenAI chat model and returns the response.
    Args:
        prompt (str): The user input or query to send to the model.
        model (str): The name of the OpenAI model to use (default is "gpt-4o").
    Returns:
        str: The assistant's textual response from the model.
    """
    from openai import OpenAI
    openai_client = OpenAI()
    # Construct the message list with a single user message
    messages = [{"role": "user", "content": prompt}]
    # Make the API call to OpenAI's chat completion endpoint
    response = openai_client.chat.completions.create(
        model=model,          # Specify which model to use (e.g., gpt-4o, gpt-4, gpt-3.5-turbo)
        messages=messages,    # The message history to provide context (only one user message here)
        temperature=0         # Controls randomness: 0 = deterministic, 1 = creative/random
    )
    # Extract and return the generated response content from the first choice
    return response.choices[0].message.content

###Implementing Few shot prompting

In [ ]:
prompt = f"""
You are a medical triage assistant. Read each HCP query and classify it into one of the following:
["Adverse Event", "Product Inquiry", "Off-label Use Inquiry", "Medical Literature Request", "Clinical Trial Request", "Other"]
Examples:
Q: "Patient developed severe rash after starting DrugX. Is this expected?"
A: Adverse Event
Q: "Can you provide the mechanism of action of DrugY?"
A: Product Inquiry
Q: "Is DrugZ effective in pediatric patients with rheumatoid arthritis?"
A: Off-label Use Inquiry
Q: "Please send me publications related to DrugA's efficacy in migraines."
A: Medical Literature Request
Q: "How do I enroll my patients into the ongoing phase 3 trial for DrugB?"
A: Clinical Trial Request
Q: "Where can I find pricing information for DrugC?"
A: Other

Now classify the following:
Q: "My patient experienced dizziness after taking the first dose of DrugP. Should I report it?"
"""
response = get_completion(prompt)
print(response)

Adverse Event


Principle 2

In [ ]:
clinical_trial_text = f"""
A recent Phase IIb study evaluated the investigational drug GLX-108 in 280 patients with moderate-to
severe asthma.
The randomized, double-blind trial was conducted over 16 weeks across 12 sites in Europe.
Primary outcome: improvement in FEV1 (forced expiratory volume).
The trial met its endpoint, showing a significant improvement in lung function with a favorable safety
profile.
"""
# Pharma-style multi-step prompt
prompt_1 = f"""
Perform the following actions:
1 - Summarize the following clinical trial text (delimited by triple backticks) into a single sentence.
2 - Translate the summary into Spanish.
3 - List each drug or compound name mentioned in the Spanish summary.
4 - Output a JSON object with the following keys: spanish_summary, drug_names.
Text:
```{clinical_trial_text}```
"""
response = get_completion(prompt_1)
print("Completion for prompt 1:")
print(response)

Completion for prompt 1:
1. Summary: A Phase IIb study showed that the investigational drug GLX-108 significantly improved lung function in patients with moderate-to-severe asthma over 16 weeks.

2. Spanish Translation: Un estudio de fase IIb mostró que el fármaco en investigación GLX-108 mejoró significativamente la función pulmonar en pacientes con asma moderada a severa durante 16 semanas.

3. Drug Names: GLX-108

4. JSON Object:
```json
{
  "spanish_summary": "Un estudio de fase IIb mostró que el fármaco en investigación GLX-108 mejoró significativamente la función pulmonar en pacientes con asma moderada a severa durante 16 semanas.",
  "drug_names": ["GLX-108"]
}
```


EX2

In [ ]:
# Define clinical text input
text = f"""
The Phase III trial of the drug Lorafenib, developed by BioThera, enrolled 600 patients with late-stage melanoma.
The study demonstrated a 32% improvement in progression-free survival compared to the control arm.
No new safety concerns were identified during the 18-month follow-up.
"""
# Define multi-step prompt
prompt = f"""
Your task is to perform the following actions:
1 - Summarize the following text delimited by < > using a single concise sentence.
2 - Translate the summary into French.
3 - Identify all proper names (e.g., drug names, companies, or diseases) in the French summary.
4 - Output a JSON object with the following fields:
   - summary: French translated summary
   - num_names: total count of identified names
   - schema: a list of dictionaries describing each field in the JSON, containing:
     * field: the field name
     * description: explanation of what the field contains
     * datatype: data type (e.g., string, integer)
Use the following format:
Text: <text to summarize>
Summary: <summary>
Translation: <summary translation>
Names: <list of proper names in French summary>
Output JSON: <json with summary, num_names, schema<field_of_json, description_of_field, datatype>>
Text: <{text}>
"""
# Call your completion function (assume get_completion uses OpenAI or similar)
response = get_completion(prompt)
print("\nCompletion for prompt:")
print(response)


Completion for prompt:
Summary: A Phase III trial of Lorafenib by BioThera showed a 32% improvement in progression-free survival for late-stage melanoma patients without new safety concerns.

Translation: Un essai de phase III de Lorafenib par BioThera a montré une amélioration de 32 % de la survie sans progression pour les patients atteints de mélanome à un stade avancé sans nouveaux problèmes de sécurité.

Names: Lorafenib, BioThera

Output JSON: {
  "summary": "Un essai de phase III de Lorafenib par BioThera a montré une amélioration de 32 % de la survie sans progression pour les patients atteints de mélanome à un stade avancé sans nouveaux problèmes de sécurité.",
  "num_names": 2,
  "schema": [
    {
      "field": "summary",
      "description": "French translated summary",
      "datatype": "string"
    },
    {
      "field": "num_names",
      "description": "total count of identified names",
      "datatype": "integer"
    },
    {
      "field": "schema",
      "description

####Making our LLm to create thier solution first and then compare with given data and provide the conclusion

In [ ]:
prompt = f"""
Your task is to determine if the student's solution \
is correct or not.
To solve the problem do the following:- First, work out your own solution to the problem including the final total.- Then compare your solution to the student's solution \
and evaluate if the student's solution is correct or not.
Don't decide if the student's solution is correct until
you have done the problem yourself.
Use the following format:
Question:
```
question here
```
Student's solution:
```
student's solution here
```
Actual solution:
```
steps to work out the solution and your solution here
```
Is the student's solution the same as actual solution \
just calculated:
```
yes or no
```
Student grade:
```
correct or incorrect
```
Question:
```
I'm building a solar power installation and I need help \
working out the financials.- Land costs $100 / square foot- I can buy solar panels for $250 / square foot- I negotiated a contract for maintenance that will cost \
me a flat $100k per year, and an additional $10 / square \
foot
What is the total cost for the first year of operations \
as a function of the number of square feet.
```
Student's solution:
```
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
```
Actual solution:
"""
response = get_completion(prompt)
print(response)

```
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x (since land costs $100 per square foot)
2. Solar panel cost: 250x (since solar panels cost $250 per square foot)
3. Maintenance cost: 100,000 + 10x (since the maintenance contract is $100k per year plus $10 per square foot)

Total cost for the first year:
= Land cost + Solar panel cost + Maintenance cost
= 100x + 250x + 100,000 + 10x
= 360x + 100,000
```
Is the student's solution the same as actual solution just calculated:
```
No
```
Student grade:
```
Incorrect
```
